<h1>TASK-1</h1>

In [1]:
import random
import pandas as pd
from datetime import datetime, timedelta

random.seed(42)

products = ["Laptop","Phone","Tablet","Headphones","Camera","Keyboard"]
countries = ["usa","germany","france","canada","japan","italy"]

orders = []

for i in range(185):
    orders.append({
        "order_id": i+1,
        "customer_id": random.randint(1,30),
        "product_name": random.choice(products),
        "quantity": random.randint(1,5),
        "unit_price": round(random.uniform(5,200),2),
        "order_date": (datetime(2024,1,1)+timedelta(days=random.randint(0,200))).strftime("%Y-%m-%d"),
        "shipping_country": random.choice(countries)
    })

orders += [
{"order_id":9001,"customer_id":2,"product_name":None,"quantity":2,"unit_price":20,"order_date":"2024-01-02","shipping_country":"usa"},
{"order_id":9002,"customer_id":3,"product_name":"","quantity":2,"unit_price":20,"order_date":"2024-01-02","shipping_country":"usa"},
{"order_id":9003,"customer_id":4,"product_name":None,"quantity":2,"unit_price":20,"order_date":"2024-01-02","shipping_country":"usa"},

{"order_id":9004,"customer_id":5,"product_name":"Phone","quantity":-2,"unit_price":30,"order_date":"2024-02-01","shipping_country":"usa"},
{"order_id":9005,"customer_id":5,"product_name":"Phone","quantity":2,"unit_price":-30,"order_date":"2024-02-01","shipping_country":"usa"},
{"order_id":9006,"customer_id":5,"product_name":"Phone","quantity":-1,"unit_price":-30,"order_date":"2024-02-01","shipping_country":"usa"},

{"order_id":9007,"customer_id":6,"product_name":"Laptop","quantity":2,"unit_price":100,"order_date":"not-a-date","shipping_country":"usa"},
{"order_id":9008,"customer_id":6,"product_name":"Laptop","quantity":2,"unit_price":100,"order_date":"2025-13-45","shipping_country":"usa"},
{"order_id":9009,"customer_id":6,"product_name":"Laptop","quantity":2,"unit_price":100,"order_date":"2024-99-10","shipping_country":"usa"},

{"order_id":10,"customer_id":7,"product_name":"Tablet","quantity":2,"unit_price":40,"order_date":"2024-03-02","shipping_country":"usa"},
{"order_id":11,"customer_id":7,"product_name":"Tablet","quantity":2,"unit_price":40,"order_date":"2024-03-02","shipping_country":"usa"},
{"order_id":12,"customer_id":7,"product_name":"Tablet","quantity":2,"unit_price":40,"order_date":"2024-03-02","shipping_country":"usa"},

{"order_id":9010,"customer_id":8,"product_name":"Camera","quantity":"3","unit_price":50,"order_date":"2024-04-01","shipping_country":"usa"},
{"order_id":9011,"customer_id":8,"product_name":"Camera","quantity":3,"unit_price":"50","order_date":"2024-04-01","shipping_country":"usa"},
{"order_id":9012,"customer_id":8,"product_name":"Camera","quantity":"two","unit_price":"fifty","order_date":"2024-04-01","shipping_country":"usa"}
]

orders

[{'order_id': 1,
  'customer_id': 21,
  'product_name': 'Laptop',
  'quantity': 1,
  'unit_price': 149.6,
  'order_date': '2024-03-03',
  'shipping_country': 'germany'},
 {'order_id': 2,
  'customer_id': 5,
  'product_name': 'Keyboard',
  'quantity': 1,
  'unit_price': 136.96,
  'order_date': '2024-05-19',
  'shipping_country': 'usa'},
 {'order_id': 3,
  'customer_id': 19,
  'product_name': 'Headphones',
  'quantity': 1,
  'unit_price': 10.81,
  'order_date': '2024-02-25',
  'shipping_country': 'germany'},
 {'order_id': 4,
  'customer_id': 17,
  'product_name': 'Camera',
  'quantity': 1,
  'unit_price': 114.44,
  'order_date': '2024-07-02',
  'shipping_country': 'italy'},
 {'order_id': 5,
  'customer_id': 23,
  'product_name': 'Camera',
  'quantity': 4,
  'unit_price': 47.99,
  'order_date': '2024-05-30',
  'shipping_country': 'france'},
 {'order_id': 6,
  'customer_id': 26,
  'product_name': 'Laptop',
  'quantity': 2,
  'unit_price': 141.14,
  'order_date': '2024-03-28',
  'shipping_c

In [2]:
from datetime import datetime

def extract(raw_records):
    valid_records = []
    rejected_records = []
    reasons = {}

    for r in raw_records:
        reason = None

        required = ["order_id","customer_id","product_name","quantity","unit_price","order_date","shipping_country"]
        for field in required:
            if field not in r:
                reason = "missing field"
                break

        if not reason:
            if r["product_name"] is None or r["product_name"] == "":
                reason = "missing product_name"

        if not reason:
            try:
                qty = float(r["quantity"])
                price = float(r["unit_price"])
                if qty <= 0 or price <= 0:
                    reason = "non-positive quantity or price"
            except:
                reason = "quantity or price not numeric"

        if not reason:
            try:
                datetime.strptime(r["order_date"], "%Y-%m-%d")
            except:
                reason = "invalid order_date"

        if reason:
            rec = r.copy()
            rec["reason"] = reason
            rejected_records.append(rec)
            reasons[reason] = reasons.get(reason, 0) + 1
        else:
            clean = r.copy()
            clean["quantity"] = float(r["quantity"])
            clean["unit_price"] = float(r["unit_price"])
            valid_records.append(clean)

    print("Valid records:", len(valid_records))
    print("Rejected records:", len(rejected_records))
    print("Rejected by reason:")
    for k,v in reasons.items():
        print(k, ":", v)

    return valid_records, rejected_records

valid_records, rejected_records = extract(orders)

Valid records: 190
Rejected records: 10
Rejected by reason:
missing product_name : 3
non-positive quantity or price : 3
invalid order_date : 3
quantity or price not numeric : 1


In [3]:
import pandas as pd

def transform(valid_records):
    df = pd.DataFrame(valid_records)

    df["total_amount"] = df["quantity"] * df["unit_price"]

    df["order_date"] = pd.to_datetime(df["order_date"])
    df["order_month"] = df["order_date"].dt.month
    df["order_day_of_week"] = df["order_date"].dt.day_name()

    df["shipping_country"] = df["shipping_country"].str.title()

    df = df.drop_duplicates(subset="order_id", keep="first")

    def category(x):
        if x < 25:
            return "small"
        elif x <= 100:
            return "medium"
        else:
            return "large"

    df["amount_category"] = df["total_amount"].apply(category)

    return df
    
clean_df = transform(valid_records)

print(clean_df.head())
print(clean_df.info())

   order_id  customer_id product_name  quantity  unit_price order_date  \
0         1           21       Laptop       1.0      149.60 2024-03-03   
1         2            5     Keyboard       1.0      136.96 2024-05-19   
2         3           19   Headphones       1.0       10.81 2024-02-25   
3         4           17       Camera       1.0      114.44 2024-07-02   
4         5           23       Camera       4.0       47.99 2024-05-30   

  shipping_country  total_amount  order_month order_day_of_week  \
0          Germany        149.60            3            Sunday   
1              Usa        136.96            5            Sunday   
2          Germany         10.81            2            Sunday   
3            Italy        114.44            7           Tuesday   
4           France        191.96            5          Thursday   

  amount_category  
0           large  
1           large  
2           small  
3           large  
4           large  
<class 'pandas.core.frame.DataFr

In [4]:
import pandas as pd

def load(df, path):
    df.to_parquet(path, index=False)

    df_loaded = pd.read_parquet(path)

    print("Original rows:", len(df))
    print("Loaded rows:", len(df_loaded))

    print("Original dtypes:")
    print(df.dtypes)

    print("\nLoaded dtypes:")
    print(df_loaded.dtypes)

    if len(df) == len(df_loaded) and all(df.dtypes == df_loaded.dtypes):
        print("Verification successful")
    else:
        print("Verification failed")

    return df_loaded

df_final = load(clean_df, "orders_clean.parquet")

Original rows: 187
Loaded rows: 187
Original dtypes:
order_id                      int64
customer_id                   int64
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month                   int32
order_day_of_week            object
amount_category              object
dtype: object

Loaded dtypes:
order_id                      int64
customer_id                   int64
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month                   int32
order_day_of_week            object
amount_category              object
dtype: object
Verification successful


In [5]:
def run_pipeline(raw_records, output_path):
    print("Running ETL pipeline...\n")

    raw_count = len(raw_records)

    valid_records, rejected_records = extract(raw_records)
    valid_count = len(valid_records)

    df = transform(valid_records)
    transformed_count = len(df)

    df_loaded = load(df, output_path)
    loaded_count = len(df_loaded)

    print("Raw records:", raw_count)
    print("Valid records:", valid_count)
    print("Rejected records:", len(rejected_records))
    print("Transformed records:", transformed_count)
    print("Loaded records:", loaded_count)

    return df_loaded

final_df = run_pipeline(orders, "orders_clean.parquet")

Running ETL pipeline...

Valid records: 190
Rejected records: 10
Rejected by reason:
missing product_name : 3
non-positive quantity or price : 3
invalid order_date : 3
quantity or price not numeric : 1
Original rows: 187
Loaded rows: 187
Original dtypes:
order_id                      int64
customer_id                   int64
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month                   int32
order_day_of_week            object
amount_category              object
dtype: object

Loaded dtypes:
order_id                      int64
customer_id                   int64
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month                   i

<h1>TASK-2</h1>

In [6]:
import pandas as pd
import json

def extract_elt(raw_records):
    good = []
    rejected = []

    for r in raw_records:
        try:
            if isinstance(r, dict):
                good.append(r)
            else:
                obj = json.loads(r)
                good.append(obj)
        except Exception:
            rejected.append(r)

    print("Extract stage")
    print("Accepted records:", len(good))
    print("Rejected records:", len(rejected))

    return good


def load_elt(records, path):
    df = pd.DataFrame(records)
    df.to_parquet(path, index=False)

    print("\nLoad stage")
    print("Saved to data lake:", path)
    print("Rows saved:", len(df))


def transform_elt(path):
    df = pd.read_parquet(path)

    df = df.dropna(subset=["product_name"])

    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)]

    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df = df.dropna(subset=["order_date"])

    df["shipping_country"] = df["shipping_country"].str.title()

    df["total_amount"] = df["quantity"] * df["unit_price"]

    df["order_month"] = df["order_date"].dt.month
    df["order_day_of_week"] = df["order_date"].dt.day_name()

    df = df.drop_duplicates(subset="order_id", keep="first")

    def category(x):
        if x < 25:
            return "small"
        elif x <= 100:
            return "medium"
        else:
            return "large"

    df["amount_category"] = df["total_amount"].apply(category)

    print("\nTransform stage")
    print("Clean rows:", len(df))

    return df

1) In the ETL approach, only the cleaned and validated records reached the final dataset because errors were filtered before loading. In ELT, almost all raw records were first stored in the data lake, and the problematic ones were removed later during transformation.

2) ETL caught most issues during the extract stage before loading. ELT detected and fixed problems during the transform stage after the raw data was already stored.

3) ETL ensures cleaner data before storage and reduces bad data in the system. ELT keeps raw data for future reprocessing and is more flexible when transformation rules change.

4) ETL is better when strict data quality is needed before loading. ELT is preferred in modern data lakes or cloud warehouses where storing raw data is useful for later analysis.

<h1>TASK-3</h1>

In [7]:
import datetime

db = {
    "orders": [],
    "features": {}
}

def order_ingestion_service(order):
    db["orders"].append(order)

def feature_computation_service():
    customer_data = {}

    for o in db["orders"]:
        cid = o["customer_id"]
        amount = o["quantity"] * o["unit_price"]
        date = datetime.datetime.strptime(o["order_date"], "%Y-%m-%d")

        if cid not in customer_data:
            customer_data[cid] = {
                "total_orders": 0,
                "total_amount": 0,
                "last_order_date": date
            }

        customer_data[cid]["total_orders"] += 1
        customer_data[cid]["total_amount"] += amount

        if date > customer_data[cid]["last_order_date"]:
            customer_data[cid]["last_order_date"] = date

    for cid, data in customer_data.items():
        avg = data["total_amount"] / data["total_orders"]

        db["features"][cid] = {
            "total_orders": data["total_orders"],
            "avg_amount": avg,
            "last_order_date": data["last_order_date"]
        }

def prediction_service(customer_id):
    features = db["features"].get(customer_id)

    if not features:
        return "No data"

    if features["avg_amount"] > 100:
        return "High value customer"
    else:
        return "Regular customer"


order_ingestion_service({
    "customer_id": 1,
    "quantity": 2,
    "unit_price": 50,
    "order_date": "2025-03-01"
})

order_ingestion_service({
    "customer_id": 1,
    "quantity": 1,
    "unit_price": 200,
    "order_date": "2025-03-05"
})

order_ingestion_service({
    "customer_id": 2,
    "quantity": 3,
    "unit_price": 20,
    "order_date": "2025-03-03"
})

feature_computation_service()

print(db["features"])
print(prediction_service(1))
print(prediction_service(2))

{1: {'total_orders': 2, 'avg_amount': 150.0, 'last_order_date': datetime.datetime(2025, 3, 5, 0, 0)}, 2: {'total_orders': 1, 'avg_amount': 60.0, 'last_order_date': datetime.datetime(2025, 3, 3, 0, 0)}}
High value customer
Regular customer


In [8]:
import datetime

orders_db = []

def order_service_add(order):
    orders_db.append(order)

def order_service_get(customer_id):
    return [o for o in orders_db if o["customer_id"] == customer_id]


def feature_service(customer_id):
    orders = order_service_get(customer_id)

    if not orders:
        return None

    total_orders = 0
    total_amount = 0
    last_date = None

    for o in orders:
        amount = o["quantity"] * o["unit_price"]
        date = datetime.datetime.strptime(o["order_date"], "%Y-%m-%d")

        total_orders += 1
        total_amount += amount

        if last_date is None or date > last_date:
            last_date = date

    avg_amount = total_amount / total_orders

    return {
        "total_orders": total_orders,
        "avg_amount": avg_amount,
        "last_order_date": last_date
    }


def prediction_service(customer_id):
    features = feature_service(customer_id)

    if not features:
        return "No data"

    if features["avg_amount"] > 100:
        return "High value customer"
    else:
        return "Regular customer"


order_service_add({
    "customer_id": 1,
    "quantity": 2,
    "unit_price": 50,
    "order_date": "2025-03-01"
})

order_service_add({
    "customer_id": 1,
    "quantity": 1,
    "unit_price": 200,
    "order_date": "2025-03-05"
})

order_service_add({
    "customer_id": 2,
    "quantity": 3,
    "unit_price": 20,
    "order_date": "2025-03-03"
})


print(prediction_service(1))
print(prediction_service(2))

High value customer
Regular customer


In [9]:
import datetime

class Broker:
    def __init__(self):
        self.topics = {}
        self.subscribers = {}

    def publish(self, topic, message):
        if topic not in self.topics:
            self.topics[topic] = []
        self.topics[topic].append(message)

        if topic in self.subscribers:
            for fn in self.subscribers[topic]:
                fn(message)

    def subscribe(self, topic, fn):
        if topic not in self.subscribers:
            self.subscribers[topic] = []
        self.subscribers[topic].append(fn)

    def get_latest(self, topic):
        if topic in self.topics and self.topics[topic]:
            return self.topics[topic][-1]
        return None


broker = Broker()

customer_stats = {}


def order_service(order):
    broker.publish("orders", order)


def feature_service(order):
    cid = order["customer_id"]
    amount = order["quantity"] * order["unit_price"]
    date = datetime.datetime.strptime(order["order_date"], "%Y-%m-%d")

    if cid not in customer_stats:
        customer_stats[cid] = {
            "total_orders": 0,
            "total_amount": 0,
            "last_order_date": date
        }

    stats = customer_stats[cid]

    stats["total_orders"] += 1
    stats["total_amount"] += amount

    if date > stats["last_order_date"]:
        stats["last_order_date"] = date

    avg = stats["total_amount"] / stats["total_orders"]

    features = {
        "customer_id": cid,
        "total_orders": stats["total_orders"],
        "avg_amount": avg,
        "last_order_date": stats["last_order_date"]
    }

    broker.publish("features", features)


def prediction_service(features):
    if features["avg_amount"] > 100:
        result = "High value customer"
    else:
        result = "Regular customer"

    print("Prediction for customer", features["customer_id"], ":", result)


broker.subscribe("orders", feature_service)
broker.subscribe("features", prediction_service)


order_service({
    "customer_id": 1,
    "quantity": 2,
    "unit_price": 50,
    "order_date": "2025-03-01"
})

order_service({
    "customer_id": 1,
    "quantity": 1,
    "unit_price": 200,
    "order_date": "2025-03-05"
})

order_service({
    "customer_id": 2,
    "quantity": 3,
    "unit_price": 20,
    "order_date": "2025-03-03"
})

Prediction for customer 1 : Regular customer
Prediction for customer 1 : High value customer
Prediction for customer 2 : Regular customer


In [10]:
import random
import datetime

new_orders = []
for i in range(20):
    new_orders.append({
        "customer_id": random.randint(1, 5),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(10, 200), 2),
        "order_date": (datetime.datetime(2025, 3, 1) + datetime.timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d")
    })

db = {"orders": [], "features": {}}

def etl_order_service(order):
    db["orders"].append(order)

def etl_feature_service():
    customer_data = {}
    for o in db["orders"]:
        cid = o["customer_id"]
        amount = o["quantity"] * o["unit_price"]
        date = datetime.datetime.strptime(o["order_date"], "%Y-%m-%d")
        if cid not in customer_data:
            customer_data[cid] = {"total_orders": 0, "total_amount": 0, "last_order_date": date}
        customer_data[cid]["total_orders"] += 1
        customer_data[cid]["total_amount"] += amount
        if date > customer_data[cid]["last_order_date"]:
            customer_data[cid]["last_order_date"] = date
    for cid, data in customer_data.items():
        db["features"][cid] = {
            "total_orders": data["total_orders"],
            "avg_amount": data["total_amount"]/data["total_orders"],
            "last_order_date": data["last_order_date"]
        }

def etl_prediction_service(cid):
    f = db["features"].get(cid)
    if not f:
        return None
    return "High value" if f["avg_amount"] > 100 else "Regular"

for o in new_orders:
    etl_order_service(o)
etl_feature_service()
etl_preds = {cid: etl_prediction_service(cid) for cid in set(o["customer_id"] for o in new_orders)}

orders_db = []

def service_order_add(order):
    orders_db.append(order)

def service_feature_service(cid):
    orders = [o for o in orders_db if o["customer_id"] == cid]
    total_orders = len(orders)
    total_amount = sum(o["quantity"]*o["unit_price"] for o in orders)
    last_date = max(datetime.datetime.strptime(o["order_date"], "%Y-%m-%d") for o in orders)
    return {"total_orders": total_orders, "avg_amount": total_amount/total_orders, "last_order_date": last_date}

def service_prediction_service(cid):
    f = service_feature_service(cid)
    return "High value" if f["avg_amount"] > 100 else "Regular"

for o in new_orders:
    service_order_add(o)
service_preds = {cid: service_prediction_service(cid) for cid in set(o["customer_id"] for o in new_orders)}

class Broker:
    def __init__(self):
        self.subscribers = {}
    def publish(self, topic, msg):
        for fn in self.subscribers.get(topic, []):
            fn(msg)
    def subscribe(self, topic, fn):
        if topic not in self.subscribers:
            self.subscribers[topic] = []
        self.subscribers[topic].append(fn)

broker = Broker()
customer_stats = {}
pubsub_preds = {}

def feature_service_pubsub(order):
    cid = order["customer_id"]
    amount = order["quantity"]*order["unit_price"]
    date = datetime.datetime.strptime(order["order_date"], "%Y-%m-%d")
    if cid not in customer_stats:
        customer_stats[cid] = {"total_orders":0,"total_amount":0,"last_order_date":date}
    stats = customer_stats[cid]
    stats["total_orders"] += 1
    stats["total_amount"] += amount
    if date > stats["last_order_date"]:
        stats["last_order_date"] = date
    avg = stats["total_amount"]/stats["total_orders"]
    features = {"customer_id": cid, "total_orders": stats["total_orders"], "avg_amount": avg, "last_order_date": stats["last_order_date"]}
    broker.publish("features", features)

def prediction_service_pubsub(features):
    cid = features["customer_id"]
    pred = "High value" if features["avg_amount"]>100 else "Regular"
    pubsub_preds[cid] = pred

broker.subscribe("orders", feature_service_pubsub)
broker.subscribe("features", prediction_service_pubsub)

for o in new_orders:
    broker.publish("orders", o)

print("ETL predictions:", etl_preds)
print("Service-call predictions:", service_preds)
print("Pub/Sub predictions:", pubsub_preds)
print("All equal:", etl_preds==service_preds==pubsub_preds)

ETL predictions: {1: 'High value', 2: 'High value', 3: 'High value', 4: 'High value', 5: 'High value'}
Service-call predictions: {1: 'High value', 2: 'High value', 3: 'High value', 4: 'High value', 5: 'High value'}
Pub/Sub predictions: {5: 'High value', 4: 'High value', 3: 'High value', 2: 'High value', 1: 'High value'}
All equal: True




Coupling:  
ETL (shared DB) - tightly connected; all services depend on the same database.  
Service-call → medium coupling; services talk directly via function/API calls.  
Pub/Sub → loosely coupled; services communicate only through messages, not directly.

Latency:  
ETL - batch; predictions come after all orders are collected.  
Service-call - on-demand; latency depends on API calls.  
Pub/Sub - almost real-time; predictions happen when a new order is published.

Failure: 
ETL - if one service fails, raw data stays but features may not update.  
Service-call - if a service fails, everything depending on it fails too.  
Pub/Sub - services are independent; if a subscriber fails, it can catch up later.

Summary:  
ETL is simple for batch analytics. Service-call is good for small, synchronous systems. Pub/Sub works best for real-time systems and is more flexible for scaling.

<h1>TASK-4</h1>

In [11]:
import pandas as pd

def daily_aggregates(df):
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["total_amount"] = df["quantity"] * df["unit_price"]

    agg_list = []

    for date, group in df.groupby(df["order_date"].dt.date):
        total_orders = len(group)
        total_revenue = group["total_amount"].sum()
        avg_order_size = group["total_amount"].mean()
        unique_customers = group["customer_id"].nunique()
        top_product = group.groupby("product_name")["total_amount"].sum().idxmax()

        agg_list.append({
            "date": date,
            "total_orders": total_orders,
            "total_revenue": total_revenue,
            "avg_order_size": avg_order_size,
            "unique_customers": unique_customers,
            "top_product": top_product
        })

    return pd.DataFrame(agg_list).sort_values("date").reset_index(drop=True)

daily_summary = daily_aggregates(clean_df)
print(daily_summary)

           date  total_orders  total_revenue  avg_order_size  \
0    2024-01-01             3         964.60      321.533333   
1    2024-01-02             1         395.85      395.850000   
2    2024-01-04             1         131.20      131.200000   
3    2024-01-05             2         710.61      355.305000   
4    2024-01-06             1          87.76       87.760000   
..          ...           ...            ...             ...   
116  2024-07-12             1          64.25       64.250000   
117  2024-07-13             3        1565.36      521.786667   
118  2024-07-14             2        1535.56      767.780000   
119  2024-07-15             1         277.00      277.000000   
120  2024-07-16             2         228.81      114.405000   

     unique_customers top_product  
0                   3  Headphones  
1                   1      Laptop  
2                   1       Phone  
3                   2  Headphones  
4                   1      Laptop  
..             

In [12]:
from collections import deque, Counter

class StreamProcessor:
    def __init__(self, window_size=50):
        self.total_orders = 0
        self.total_revenue = 0
        self.unique_customers = set()
        self.window_size = window_size
        self.window = deque()
        self.product_counts = Counter()

    def process_order(self, order):
        amount = order["quantity"] * order["unit_price"]
        self.total_orders += 1
        self.total_revenue += amount
        self.unique_customers.add(order["customer_id"])

        self.window.append(order)
        self.product_counts[order["product_name"]] += 1

        if len(self.window) > self.window_size:
            old = self.window.popleft()
            self.product_counts[old["product_name"]] -= 1
            if self.product_counts[old["product_name"]] == 0:
                del self.product_counts[old["product_name"]]

        window_avg = sum(o["quantity"]*o["unit_price"] for o in self.window)/len(self.window)
        most_popular_product = self.product_counts.most_common(1)[0][0] if self.product_counts else None

        return {
            "total_orders": self.total_orders,
            "total_revenue": self.total_revenue,
            "unique_customers": len(self.unique_customers),
            "window_avg_order_size": window_avg,
            "current_top_product": most_popular_product
        }

stream = StreamProcessor(window_size=50)

for i, order in enumerate(clean_df.to_dict("records"), start=1):
    stats = stream.process_order(order)
    if i % 50 == 0:
        print(f"After {i} orders:")
        print(stats)

After 50 orders:
{'total_orders': 50, 'total_revenue': 12227.589999999998, 'unique_customers': 24, 'window_avg_order_size': 244.55180000000001, 'current_top_product': 'Keyboard'}
After 100 orders:
{'total_orders': 100, 'total_revenue': 26010.120000000003, 'unique_customers': 29, 'window_avg_order_size': 275.6506, 'current_top_product': 'Keyboard'}
After 150 orders:
{'total_orders': 150, 'total_revenue': 43595.51, 'unique_customers': 30, 'window_avg_order_size': 351.70779999999996, 'current_top_product': 'Headphones'}


When we compare cumulative totals, like total orders or total revenue, both windowed streaming and daily batch computations generally match because they are aggregating the same set of data. Each order contributes to the totals in both approaches, so the overall sums should be identical. However, windowed streaming statistics often differ from daily batch statistics when we look at specific metrics, such as averages, counts within a moving window, or recency-based features. This happens because streaming metrics only consider a limited, recent subset of data (for example, the last 50 orders), whereas batch metrics summarize all historical data for the day or reporting period. As a result, the streaming metrics can fluctuate more and react immediately to recent changes, while batch metrics are stable and represent the overall trend.

Batch statistics provide a complete view of customer behavior over time, capturing overall trends, total revenue, average order size, unique customer counts, and other aggregates. They are particularly useful for understanding historical patterns, evaluating long-term performance, or predicting long-term customer value. Decision-making based on batch data helps companies plan marketing strategies, inventory, and reporting because it reflects the full picture of what has occurred.

Streaming statistics, on the other hand, focus on short-term, recent behavior. They capture the immediate state of the system, such as spikes in orders, sudden changes in customer activity, or emerging popular products. This real-time perspective allows businesses to make quick decisions, like running flash sales, targeting active customers, or detecting anomalies in orders. While the totals eventually match batch aggregates, the streaming approach highlights trends and variations that batch processing cannot show until after the period ends.

In summary, batch processing tells you about long-term patterns and totals, while streaming processing tells you about recent trends and immediate changes. Both are valuable, and combining them provides a comprehensive understanding of both historical behavior and real-time dynamics.

<h1>TASK-5</h1>

In [13]:
import pandas as pd
from datetime import datetime

clean_df["order_date"] = pd.to_datetime(clean_df["order_date"])
clean_df["total_amount"] = clean_df["quantity"] * clean_df["unit_price"]

batch_features = []
for cid, orders in clean_df.groupby("customer_id"):
    total_lifetime_orders = len(orders)
    avg_order_amount = orders["total_amount"].mean()
    days_since_first_order = (pd.Timestamp.now() - orders["order_date"].min()).days
    most_purchased_category = orders["product_name"].value_counts().idxmax()

    batch_features.append({
        "customer_id": cid,
        "total_lifetime_orders": total_lifetime_orders,
        "avg_order_amount": avg_order_amount,
        "days_since_first_order": days_since_first_order,
        "most_purchased_category": most_purchased_category
    })

batch_features_df = pd.DataFrame(batch_features)
print(batch_features_df)

streaming_features = []
for cid, orders in clean_df.groupby("customer_id"):
    orders = orders.sort_values("order_date").tail(10)
    recent_order_count = len(orders)
    recent_avg_amount = orders["total_amount"].mean()
    seconds_since_last_order = (pd.Timestamp.now() - orders["order_date"].max()).total_seconds()
    recent_top_category = orders["product_name"].value_counts().idxmax()

    streaming_features.append({
        "customer_id": cid,
        "recent_order_count": recent_order_count,
        "recent_avg_amount": recent_avg_amount,
        "seconds_since_last_order": seconds_since_last_order,
        "recent_top_category": recent_top_category
    })

streaming_features_df = pd.DataFrame(streaming_features)
print(streaming_features_df)

    customer_id  total_lifetime_orders  avg_order_amount  \
0             1                      5        496.574000   
1             2                      5        183.118000   
2             3                      8        285.993750   
3             4                      5        503.856000   
4             5                      9        261.114444   
5             6                      1        485.200000   
6             7                      8        338.716250   
7             8                      7        128.238571   
8             9                      7        199.930000   
9            10                      6        285.565000   
10           11                      4        221.530000   
11           12                      6        315.266667   
12           13                      2        266.405000   
13           14                      6        455.206667   
14           15                      8        324.108750   
15           16                      6  

In [14]:
import pandas as pd
from datetime import datetime

clean_df["order_date"] = pd.to_datetime(clean_df["order_date"])
clean_df["total_amount"] = clean_df["quantity"] * clean_df["unit_price"]

sample_customers = clean_df["customer_id"].drop_duplicates().sample(5, random_state=42).tolist()

batch_features = []
for cid in sample_customers:
    orders = clean_df[clean_df["customer_id"] == cid]
    total_lifetime_orders = len(orders)
    avg_order_amount = orders["total_amount"].mean()
    days_since_first_order = (pd.Timestamp.now() - orders["order_date"].min()).days
    most_purchased_category = orders["product_name"].value_counts().idxmax()
    
    batch_features.append({
        "customer_id": cid,
        "total_lifetime_orders": total_lifetime_orders,
        "avg_order_amount": avg_order_amount,
        "days_since_first_order": days_since_first_order,
        "most_purchased_category": most_purchased_category
    })

batch_features_df = pd.DataFrame(batch_features)
print("Batch Features for Sample Customers:")
print(batch_features_df)

streaming_features = []
for cid in sample_customers:
    orders = clean_df[clean_df["customer_id"] == cid].sort_values("order_date").tail(10)
    recent_order_count = len(orders)
    recent_avg_amount = orders["total_amount"].mean()
    seconds_since_last_order = (pd.Timestamp.now() - orders["order_date"].max()).total_seconds()
    recent_top_category = orders["product_name"].value_counts().idxmax()
    
    streaming_features.append({
        "customer_id": cid,
        "recent_order_count": recent_order_count,
        "recent_avg_amount": recent_avg_amount,
        "seconds_since_last_order": seconds_since_last_order,
        "recent_top_category": recent_top_category
    })

streaming_features_df = pd.DataFrame(streaming_features)
print("Streaming Features for Sample Customers:")
print(streaming_features_df)

Batch Features for Sample Customers:
   customer_id  total_lifetime_orders  avg_order_amount  \
0           13                      2        266.405000   
1           30                      6        373.183333   
2           28                      6        302.078333   
3           16                      6        345.706667   
4           18                      6        289.446667   

   days_since_first_order most_purchased_category  
0                     748                  Laptop  
1                     793                  Tablet  
2                     797              Headphones  
3                     784                  Laptop  
4                     787                   Phone  
Streaming Features for Sample Customers:
   customer_id  recent_order_count  recent_avg_amount  \
0           13                   2         266.405000   
1           30                   6         373.183333   
2           28                   6         302.078333   
3           16             

In [15]:
import pandas as pd
from datetime import datetime

clean_df["order_date"] = pd.to_datetime(clean_df["order_date"])
clean_df["total_amount"] = clean_df["quantity"] * clean_df["unit_price"]

sample_customers = clean_df["customer_id"].drop_duplicates().sample(5, random_state=42).tolist()

combined_features = []

for cid in sample_customers:
    orders = clean_df[clean_df["customer_id"] == cid].sort_values("order_date")
    
    total_lifetime_orders = len(orders)
    avg_order_amount = orders["total_amount"].mean()
    days_since_first_order = (pd.Timestamp.now() - orders["order_date"].min()).days
    most_purchased_category = orders["product_name"].value_counts().idxmax()
    
    last_orders = orders.tail(10)
    recent_order_count = len(last_orders)
    recent_avg_amount = last_orders["total_amount"].mean()
    seconds_since_last_order = (pd.Timestamp.now() - last_orders["order_date"].max()).total_seconds()
    recent_top_category = last_orders["product_name"].value_counts().idxmax()
    
    combined_features.append({
        "customer_id": cid,
        "total_lifetime_orders": total_lifetime_orders,
        "avg_order_amount": avg_order_amount,
        "days_since_first_order": days_since_first_order,
        "most_purchased_category": most_purchased_category,
        "recent_order_count": recent_order_count,
        "recent_avg_amount": recent_avg_amount,
        "seconds_since_last_order": seconds_since_last_order,
        "recent_top_category": recent_top_category
    })

combined_features_df = pd.DataFrame(combined_features)
print(combined_features_df)

   customer_id  total_lifetime_orders  avg_order_amount  \
0           13                      2        266.405000   
1           30                      6        373.183333   
2           28                      6        302.078333   
3           16                      6        345.706667   
4           18                      6        289.446667   

   days_since_first_order most_purchased_category  recent_order_count  \
0                     748                Keyboard                   2   
1                     793                  Tablet                   6   
2                     797              Headphones                   6   
3                     784                  Laptop                   6   
4                     787                   Phone                   6   

   recent_avg_amount  seconds_since_last_order recent_top_category  
0         266.405000              5.883069e+07            Keyboard  
1         373.183333              5.796669e+07              Tablet  

Machine learning models work best when they use both historical trends (batch features) and recent behavior (stream features). Batch features show long-term patterns, while stream features capture immediate, short-term changes.

Example where batch features alone are insufficient: Predicting whether a customer will make a purchase today. Historical averages may indicate usual behavior, but they cannot capture a sudden surge in interest or a flash sale. Without stream features, the model would miss these short-term signals.

Example where stream features alone are insufficient: Predicting a customer’s lifetime value or long-term churn risk. Looking only at the last 50 orders may misrepresent a generally loyal or high-spending customer, because short-term behavior does not reflect overall patterns. Batch features are needed to capture long-term trends.